### bmi.csv 데이터 사용, 네이티브 케라스로 분류 모델 작성 후 정확도 출력

In [1]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten
import np_utils # pip install np_utils
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 데이터 준비 및 전처리

In [2]:
data = pd.read_csv('bmi.csv')
data.head(2)

,height,weight,label
0,142,62,fat
1,142,73,fat


In [3]:
data.describe()

,height,weight
count,20000.000000,20000.000000
mean,159.927200,57.535000
std,23.342096,13.285259
min,120.000000,35.000000
25%,140.000000,46.000000
50%,160.000000,58.000000
75%,180.000000,69.000000
max,200.000000,80.000000


In [4]:
data['label'].unique()

<StringArray>
['fat', 'normal', 'thin']
Length: 3, dtype: str

In [5]:
data.isna().sum()

height    0
weight    0
label     0
dtype: int64

In [6]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
label = le.fit_transform(data['label'])

In [17]:
from sklearn.preprocessing import MinMaxScaler

minmax = MinMaxScaler()
minmax_scaled = minmax.fit_transform(data.iloc[:,:2])
minmax_scaled[:5,:]

array([[0.275     , 0.6       ],
       [0.275     , 0.84444444],
       [0.7125    , 0.57777778],
       [0.8375    , 0.28888889],
       [0.4125    , 0.55555556]])

In [28]:
from sklearn.preprocessing import StandardScaler

stsc = StandardScaler()
standard_scaled = stsc.fit_transform(data.iloc[:,:2])
standard_scaled[:5,:]

array([[-0.76803935,  0.3360952 ],
       [-0.76803935,  1.16410127],
       [ 0.73143504,  0.26082192],
       [ 1.15985629, -0.71773072],
       [-0.29677597,  0.18554864]])

In [45]:
X_train, X_test, y_train, y_test = train_test_split(standard_scaled, label, test_size=0.1, random_state=42)
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((18000, 2), (18000,), (2000, 2), (2000,))

In [46]:
y_train

array([1, 2, 2, ..., 2, 2, 1])

In [47]:
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

# 케라스 분류 모델

In [118]:
model = Sequential()
# 입력층
model.add(Flatten(input_shape=(2,)))
model.add(Dense(8, activation='relu'))
model.add(Dropout(0.1))
# 은닉층
model.add(Dense(6, activation='relu'))
model.add(Dropout(0.1))
# 출력층
model.add(Dense(3, activation='softmax'))
model.summary()

Model: "sequential_21"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten_21 (Flatten)        (None, 2)                 0         
                                                                 
 dense_63 (Dense)            (None, 8)                 24        
                                                                 
 dropout_42 (Dropout)        (None, 8)                 0         
                                                                 
 dense_64 (Dense)            (None, 6)                 54        
                                                                 
 dropout_43 (Dropout)        (None, 6)                 0         
                                                                 
 dense_65 (Dense)            (None, 3)                 21        
                                                                 
Total params: 99 (396.00 Byte)
Trainable params: 99 (

In [119]:
model.compile(
    loss = 'categorical_crossentropy', # 다중분류, sparse_categorical_crossentropy => 정답이 정수일때
    optimizer = 'adam',
    metrics = ['accuracy'])

In [120]:
hist = model.fit(X_train, y_train, epochs=50, batch_size=32)

Epoch 1/50
563/563 [==============================] - 2s 2ms/step - loss: 0.6338 - accuracy: 0.7507
Epoch 2/50
563/563 [==============================] - 1s 2ms/step - loss: 0.2916 - accuracy: 0.8888
Epoch 3/50
563/563 [==============================] - 1s 2ms/step - loss: 0.2079 - accuracy: 0.9211
Epoch 4/50
563/563 [==============================] - 1s 2ms/step - loss: 0.1637 - accuracy: 0.9366
Epoch 5/50
563/563 [==============================] - 1s 2ms/step - loss: 0.1397 - accuracy: 0.9433
Epoch 6/50
563/563 [==============================] - 1s 2ms/step - loss: 0.1226 - accuracy: 0.9504
Epoch 7/50
563/563 [==============================] - 1s 2ms/step - loss: 0.1115 - accuracy: 0.9561
Epoch 8/50
563/563 [==============================] - 1s 2ms/step - loss: 0.0996 - accuracy: 0.9586
Epoch 9/50
563/563 [==============================] - 1s 2ms/step - loss: 0.0976 - accuracy: 0.9611
Epoch 10/50
563/563 [==============================] - 1s 2ms/step - loss: 0.0892 - accuracy: 0.9621

In [121]:
score = model.evaluate(X_test, y_test, verbose=1)
score

63/63 [==============================] - 0s 2ms/step - loss: 0.0165 - accuracy: 0.9965


[0.01646958850324154, 0.9965000152587891]